# Blinded public-video timing validation

This notebook executes the prespecified comparison between feed **event-start intervals** and independently annotated video release-to-release intervals. It does not call feed timestamps release times, fit a runner-threat model, or make causal claims. With no usable annotations, the only valid gate result is `validation_pending`.


## Frozen definitions and gate

The protocol in `docs/video_timing_validation_protocol.md` was frozen before video timing errors were observed. A release is the first displayed frame with the pitched ball visibly separated from the pitcher's hand. Both releases must appear in one verified-continuous source. Edited highlights, condensed games, separate pitch clips, and unverifiable cuts are excluded. The annotator-facing CSV contains no feed timestamps, feed intervals, or error fields; feed values are joined afterward by `(game_pk, at_bat_index, preceding_event_index, following_event_index)`.


In [1]:
from datetime import datetime, timezone
from pathlib import Path
import platform
import sys

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.audit_feeds import build_intervals, extract_events, load_feeds
from scripts.video_validation import (
    ANNOTATION_COLUMNS, PRIMARY_PER_STRATUM, STRATA, cluster_bootstrap, eligible_annotations,
    error_summary, evaluate_gate, join_annotations_to_feed,
)

executed_utc = datetime.now(timezone.utc).isoformat(timespec='seconds').replace('+00:00', 'Z')
environment = pd.DataFrame([
    {'package': 'Python', 'version': platform.python_version()},
    {'package': 'pandas', 'version': pd.__version__},
    {'package': 'numpy', 'version': np.__version__},
    {'package': 'matplotlib', 'version': matplotlib.__version__},
])
print(f'Executed UTC: {executed_utc}')
display(environment)


Executed UTC: 2026-11-15T21:39:44Z


,package,version
0,Python,3.14.4
1,pandas,3.0.2
2,numpy,2.4.4
3,matplotlib,3.10.8


In [2]:
DATA = ROOT / 'data'
RESULTS = ROOT / 'results'
candidates = pd.read_csv(DATA / 'video_validation_candidates.csv')
annotations = pd.read_csv(DATA / 'video_validation_annotations.csv', keep_default_na=False)
availability = pd.read_csv(DATA / 'video_availability_log.csv', keep_default_na=False)
runner_state = pd.read_csv(DATA / 'runner_state_validation_sample.csv', keep_default_na=False)
manifest, loaded_feeds = load_feeds(ROOT)
events, state_audit = extract_events(loaded_feeds)
intervals = build_intervals(events)

assert len(loaded_feeds) == 30, 'All 30 cached audit feeds are required.'
assert len(candidates) == 114 and candidates['validation_id'].is_unique
assert candidates['sample_role'].value_counts().to_dict() == {'primary': 90, 'reserve': 24}
stratum_counts = candidates.groupby(['sample_role', 'validation_stratum']).size()
assert all(
    count == (PRIMARY_PER_STRATUM[stratum] if role == 'primary' else 3)
    for (role, stratum), count in stratum_counts.items()
)
assert list(annotations.columns) == ANNOTATION_COLUMNS
for forbidden in ('event_start_interval_seconds', 'start_time', 'signed_error_seconds', 'absolute_error_seconds'):
    assert forbidden not in annotations.columns, f'Blinding violation: {forbidden}'
assert set(annotations['validation_id']) == set(candidates['validation_id'])
assert set(availability['validation_id']) == set(candidates['validation_id'])

sample_summary = pd.DataFrame([{
    'source_intervals': len(intervals),
    'primary_candidates': int(candidates['sample_role'].eq('primary').sum()),
    'reserve_candidates': int(candidates['sample_role'].eq('reserve').sum()),
    'primary_games': int(candidates.loc[candidates['sample_role'].eq('primary'), 'game_pk'].nunique()),
    'primary_venues': int(candidates.loc[candidates['sample_role'].eq('primary'), 'venue'].nunique()),
    'second_annotation_flags': int(candidates['second_annotation_requested'].fillna(False).sum()),
}])
display(sample_summary)
display(candidates.groupby(['sample_role', 'validation_stratum']).size().rename('intervals').reset_index())


,source_intervals,primary_candidates,reserve_candidates,primary_games,primary_venues,second_annotation_flags
0,6532,90,24,29,21,12


,sample_role,validation_stratum,intervals
0,primary,after_foul,10
1,primary,ordinary_bases_empty,15
2,primary,ordinary_runner_first_only,15
3,primary,other_interruption,10
4,primary,pickoff,10
5,primary,step_off,10
6,primary,timeout,10
7,primary,unusually_long,10
8,reserve,after_foul,3
9,reserve,ordinary_bases_empty,3


In [3]:
availability_summary = (
    availability.groupby(['sample_role', 'search_status'], dropna=False)
    .size().rename('candidate_records').reset_index()
)
source_candidates_path = DATA / 'public_video_source_candidates.csv'
source_candidates = (
    pd.read_csv(source_candidates_path, keep_default_na=False)
    if source_candidates_path.exists() else pd.DataFrame()
)
display(availability_summary)
print(f'Continuous-title public sources found in metadata: {len(source_candidates)}')
print('These are not usable unless they match a frozen candidate and continuity is visually verified.')


,sample_role,search_status,candidate_records
0,primary,inaccessible,83
1,primary,not_found,7
2,reserve,inaccessible,23
3,reserve,not_found,1


Continuous-title public sources found in metadata: 1
These are not usable unless they match a frozen candidate and continuity is visually verified.


In [4]:
joined = join_annotations_to_feed(annotations, candidates, intervals)
eligible, excluded = eligible_annotations(joined)
secondary, secondary_excluded = eligible_annotations(
    joined, annotation_rounds=('independent_second', 'repeat_same_reviewer')
)
overall_summary = error_summary(eligible)
bootstrap_summary = cluster_bootstrap(eligible, replicates=2000)
gate = evaluate_gate(eligible)

group_rows = []
for dimension in ('game_pk', 'base_state', 'validation_stratum'):
    for group_value, group in eligible.groupby(dimension, dropna=False):
        metrics = error_summary(group).iloc[0].to_dict()
        group_rows.append({'dimension': dimension, 'group': group_value, **metrics})
group_columns = [
    'dimension', 'group', 'intervals', 'games', 'mean_signed_error',
    'median_signed_error', 'mean_absolute_error', 'median_absolute_error',
    'p95_absolute_error',
]
group_summary = pd.DataFrame(group_rows, columns=group_columns)
review_pairs = eligible[[
    'validation_id', 'derived_video_interval_seconds', 'annotator'
]].merge(
    secondary[[
        'validation_id', 'derived_video_interval_seconds', 'annotator', 'annotation_round'
    ]],
    on='validation_id', how='inner', suffixes=('_primary', '_secondary'),
)
review_pairs['absolute_video_difference_seconds'] = (
    review_pairs['derived_video_interval_seconds_primary']
    - review_pairs['derived_video_interval_seconds_secondary']
).abs()
review_columns = [
    'annotation_round', 'paired_intervals', 'mean_absolute_video_difference',
    'median_absolute_video_difference', 'p95_absolute_video_difference',
]
review_rows = []
for annotation_round, group in review_pairs.groupby('annotation_round'):
    difference = group['absolute_video_difference_seconds']
    review_rows.append({
        'annotation_round': annotation_round,
        'paired_intervals': len(group),
        'mean_absolute_video_difference': difference.mean(),
        'median_absolute_video_difference': difference.median(),
        'p95_absolute_video_difference': difference.quantile(0.95),
    })
second_annotation_summary = pd.DataFrame(review_rows, columns=review_columns)
gate_summary = pd.DataFrame([{
    'executed_utc': executed_utc,
    'outcome': gate.outcome,
    'validated_scope': gate.validated_scope,
    'eligible_primary_intervals': len(eligible),
    'eligible_games': eligible['game_pk'].nunique(),
    'excluded_or_incomplete_annotation_rows': len(excluded),
    'reasons': ' | '.join(gate.reasons),
}])
display(gate_summary)
if eligible.empty:
    print('Validation pending: no genuine usable video annotations are present; no error estimate is reported.')
else:
    display(overall_summary)
    display(group_summary)
    display(bootstrap_summary)
if second_annotation_summary.empty:
    print('No usable paired second annotations are available for review-agreement summaries.')
else:
    display(second_annotation_summary)


,executed_utc,outcome,validated_scope,eligible_primary_intervals,eligible_games,excluded_or_incomplete_annotation_rows,reasons
0,2026-11-15T21:39:44Z,validation_pending,none,0,0,114,No usable primary annotations are available.


Validation pending: no genuine usable video annotations are present; no error estimate is reported.
No usable paired second annotations are available for review-agreement summaries.


In [5]:
if eligible.empty:
    print('Agreement and error plots are intentionally not drawn without usable annotations.')
else:
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.scatter(eligible['derived_video_interval_seconds'], eligible['event_start_interval_seconds'], alpha=0.7)
    upper = max(eligible['derived_video_interval_seconds'].max(), eligible['event_start_interval_seconds'].max())
    ax.plot([0, upper], [0, upper], color='black', linestyle='--', linewidth=1)
    ax.set(xlabel='Video release interval (s)', ylabel='Feed event-start interval (s)', title='Feed versus video interval')
    fig.tight_layout()
    fig.savefig(RESULTS / 'video_timing_agreement.png', dpi=150)
    plt.show()

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].hist(eligible['signed_error_seconds'], bins='auto')
    axes[0].set(xlabel='Signed error: feed - video (s)', ylabel='Intervals')
    axes[1].hist(eligible['absolute_error_seconds'], bins='auto')
    axes[1].set(xlabel='Absolute error (s)', ylabel='Intervals')
    fig.tight_layout()
    fig.savefig(RESULTS / 'video_timing_error_distributions.png', dpi=150)
    plt.show()


Agreement and error plots are intentionally not drawn without usable annotations.


In [6]:
runner_state_status = (
    runner_state.groupby(['review_status', 'sample_case'], dropna=False)
    .size().rename('cases').reset_index()
)
runner_state_overall = pd.DataFrame([{
    'cases': len(runner_state),
    'reviewed_cases': int(runner_state['review_status'].ne('pending').sum()),
    'pending_cases': int(runner_state['review_status'].eq('pending').sum()),
    'unresolved_state_flags': int(runner_state['unresolved_state_flag'].astype(str).str.lower().eq('true').sum()),
    'gate_status': 'validation_pending' if runner_state['review_status'].eq('pending').any() else 'review_complete',
}])
display(runner_state_overall)
display(runner_state_status)
display(runner_state.loc[runner_state['unresolved_state_flag'].astype(str).str.lower().eq('true'), [
    'runner_validation_id', 'game_pk', 'at_bat_index', 'notes',
]])
print('Runner-state review is a separate gate; timing agreement cannot validate it.')


,cases,reviewed_cases,pending_cases,unresolved_state_flags,gate_status
0,21,0,21,1,validation_pending


,review_status,sample_case,cases
0,pending,caught_stealing,2
1,pending,ends_without_next_pitch,1
2,pending,ordinary_nonattempt,10
3,pending,pickoff_then_pitch,4
4,pending,stolen_base,3
5,pending,terminal_state_conflict,1


,runner_validation_id,game_pk,at_bat_index,notes
20,runner_g746690_pa38_terminal_conflict,746690,38,Feed omits the terminal 1B-to-2B advance and p...


Runner-state review is a separate gate; timing agreement cannot validate it.


In [7]:
RESULTS.mkdir(exist_ok=True)
gate_summary.to_csv(RESULTS / 'video_timing_validation_summary.csv', index=False)
group_summary.to_csv(RESULTS / 'video_timing_validation_group_summary.csv', index=False)
bootstrap_summary.to_csv(RESULTS / 'video_timing_validation_bootstrap.csv', index=False)
second_annotation_summary.to_csv(RESULTS / 'video_second_annotation_summary.csv', index=False)
availability_summary.to_csv(RESULTS / 'video_timing_validation_availability_summary.csv', index=False)
runner_state_overall.to_csv(RESULTS / 'runner_state_validation_status.csv', index=False)
environment.assign(executed_utc=executed_utc).to_csv(RESULTS / 'video_validation_environment.csv', index=False)
print('Wrote compact, reviewed validation-status outputs. No raw video or full event table was written.')


Wrote compact, reviewed validation-status outputs. No raw video or full event table was written.


## Interpretation boundary

A pending gate is not a failed timing measure and is not evidence that event-start timestamps equal ball release. Only genuine blinded annotations from verified-continuous footage can change this result. A future timing pass would still not validate runner states, threat prediction, strategic value, or causal effects.
